# Paper Bag Analysis – STFT, Spectral Features & Onset Timeline

Dieses Notebook lädt eine WAV-Datei (dein Papiersackerl) und berechnet:
- STFT-Spektrogramm (dB)
- *Spectral Centroid* (Schwerpunkt)
- *Spectral Rolloff* (85% und 95%)
- *Spectral Flatness* (0..1)
- Onset-Timeline (Ereignis-Markierungen)

**Hinweise**
- Keine Subplots; jede Grafik steht separat.
- Nur Matplotlib; keine Seaborn-Abhängigkeit.
- FFT-Parameter so gewählt, dass ~75% Overlap entsteht (n_fft=2048, hop_length=512).


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf  # optional: Schreiben/Lesen von Audiodateien
print("librosa:", librosa.__version__)


In [ ]:
# Pfad & Parameter
AUDIO_PATH = "./paper_bag_crumple.wav"  # <-- Pfad anpassen
SR_TARGET = None  # None = Original-Samplerate

# STFT-Parameter (Hann, ~75% Overlap)
N_FFT = 2048
HOP_LENGTH = 512
WIN = "hann"

# Rolloff-Perzentile
ROLLOFF_PCTS = (0.85, 0.95)


In [ ]:
# Audio laden
y, sr = librosa.load(AUDIO_PATH, sr=SR_TARGET, mono=True)
duration = len(y)/sr
print(f"Geladen: {AUDIO_PATH} | sr={sr} Hz | Dauer = {duration:.2f} s | Samples = {len(y)}")


In [ ]:
# STFT & Spektrogramm (in dB)
S = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH, window=WIN)
S_mag = np.abs(S)
S_db = librosa.amplitude_to_db(S_mag, ref=np.max)
times = librosa.times_like(S, sr=sr, hop_length=HOP_LENGTH)
freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

plt.figure(figsize=(10,4))
librosa.display.specshow(S_db, x_axis='time', y_axis='hz', sr=sr, hop_length=HOP_LENGTH)
plt.title('STFT-Spektrogramm (dB)')
plt.colorbar(format='%+0.1f dB')
plt.tight_layout()
plt.show()


In [ ]:
# Spectral Centroid (Hz)
centroid = librosa.feature.spectral_centroid(S=S_mag, sr=sr)
t_cent = librosa.times_like(centroid, sr=sr, hop_length=HOP_LENGTH)
plt.figure(figsize=(10,3))
plt.plot(t_cent, centroid[0])
plt.xlabel('Zeit (s)')
plt.ylabel('Centroid (Hz)')
plt.title('Spectral Centroid')
plt.tight_layout()
plt.show()


In [ ]:
# Spectral Rolloff (85% und 95%)
rolloff_curves = {}
for pct in ROLLOFF_PCTS:
    ro = librosa.feature.spectral_rolloff(S=S_mag, sr=sr, roll_percent=pct)
    rolloff_curves[pct] = (librosa.times_like(ro, sr=sr, hop_length=HOP_LENGTH), ro[0])

plt.figure(figsize=(10,3))
for pct, (t_ro, ro_vals) in rolloff_curves.items():
    plt.plot(t_ro, ro_vals, label=f'Rolloff {int(pct*100)}%')
plt.xlabel('Zeit (s)')
plt.ylabel('Frequenz (Hz)')
plt.title('Spectral Rolloff')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Spectral Flatness (0..1)
flatness = librosa.feature.spectral_flatness(S=S_mag)
t_flat = librosa.times_like(flatness, sr=sr, hop_length=HOP_LENGTH)
plt.figure(figsize=(10,3))
plt.plot(t_flat, flatness[0])
plt.ylim(0, 1)
plt.xlabel('Zeit (s)')
plt.ylabel('Flatness (0..1)')
plt.title('Spectral Flatness')
plt.tight_layout()
plt.show()


In [ ]:
# Onset-Timeline (Energiehüllkurve + Onset-Marker)
onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP_LENGTH)
onset_times = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=HOP_LENGTH, units='time')
t_env = librosa.times_like(onset_env, sr=sr, hop_length=HOP_LENGTH)

plt.figure(figsize=(10,3))
plt.plot(t_env, onset_env)
for t in onset_times:
    plt.axvline(t, linestyle='--', alpha=0.6)
plt.xlabel('Zeit (s)')
plt.ylabel('Onset-Envelope (arb.)')
plt.title('Onset Timeline')
plt.tight_layout()
plt.show()

print(f"Anzahl Onsets: {len(onset_times)}")
print("Onset-Zeiten (s):", np.round(onset_times, 3))


## Optional: Export der Features als CSV
Praktisch, wenn du später Parameterstatistiken bilden oder Max/MSP steuern willst.

In [ ]:
import csv
EXPORT_CSV = False  # auf True setzen, um zu exportieren
CSV_PATH = 'paper_bag_features.csv'

if EXPORT_CSV:
    t_feat = t_cent  # frame-synchronisierte Zeitachse
    ro85_t, ro85 = rolloff_curves[0.85]
    ro95_t, ro95 = rolloff_curves[0.95]
    with open(CSV_PATH, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['time_s', 'centroid_hz', 'rolloff85_hz', 'rolloff95_hz', 'flatness_0_1'])
        for i in range(len(t_feat)):
            c = float(centroid[0, i]) if i < centroid.shape[1] else ''
            r85 = float(ro85[i]) if i < len(ro85) else ''
            r95 = float(ro95[i]) if i < len(ro95) else ''
            fl = float(flatness[0, i]) if i < flatness.shape[1] else ''
            w.writerow([float(t_feat[i]), c, r85, r95, fl])
    print(f"Gespeichert: {CSV_PATH}")
